# 09 Build the Entity-Resolution Knowledge Graph
This notebook converts final entity-resolution decisions into graph nodes and edges, representing input records, candidate companies, countries, manual review cases, match relationships, no-match relationships, and location relationships.

In [0]:
# Load final decisions and define graph output tables.
from pyspark.sql import functions as F

final_table = "workspace.entity_resolution_project.company_er_final_decisions"
nodes_table = "workspace.entity_resolution_project.company_er_graph_nodes"
edges_table = "workspace.entity_resolution_project.company_er_graph_edges"

final_df = spark.table(final_table)

In [0]:
# Create graph nodes for inputs, candidate companies, countries, and review queue.
# Create Node
input_nodes = (
    final_df
    .select(
        F.concat(F.lit("input:"), F.col("left_row_key").cast("string")).alias("node_id"),
        F.lit("INPUT_RECORD").alias("node_type"),
        F.col("left_company_name").alias("label"),
        F.col("left_country").alias("country"),
        F.col("left_country_code").alias("country_code"),
        F.col("left_city").alias("city")
    )
)

company_nodes = (
    final_df
    .select(
        F.concat(F.lit("company:"), F.col("right_row_key").cast("string")).alias("node_id"),
        F.lit("CANDIDATE_COMPANY").alias("node_type"),
        F.col("right_company_name").alias("label"),
        F.col("right_country").alias("country"),
        F.col("right_country_code").alias("country_code"),
        F.col("right_city").alias("city")
    )
)

country_nodes = (
    final_df
    .select(
        F.concat(F.lit("country:"), F.col("right_country_code")).alias("node_id"),
        F.lit("COUNTRY").alias("node_type"),
        F.coalesce(F.col("right_country"), F.col("right_country_code")).alias("label"),
        F.col("right_country").alias("country"),
        F.col("right_country_code").alias("country_code"),
        F.lit(None).cast("string").alias("city")
    )
    .filter(F.col("country_code").isNotNull())
)

review_node = spark.createDataFrame(
    [("review:manual_review", "REVIEW_QUEUE", "Manual Review Needed", "", "", "")],
    ["node_id", "node_type", "label", "country", "country_code", "city"]
)

nodes = (
    input_nodes
    .unionByName(company_nodes)
    .unionByName(country_nodes)
    .unionByName(review_node)
    .dropDuplicates(["node_id"])
)

In [0]:
# Create graph edges for matches, reviews, no-matches, locations, and top candidates.
# Create Edges
match_edges = (
    final_df
    .filter(F.col("final_decision") == "MATCH")
    .select(
        F.concat(F.lit("edge:match:"), F.col("left_row_key").cast("string")).alias("edge_id"),
        F.concat(F.lit("input:"), F.col("left_row_key").cast("string")).alias("source_node_id"),
        F.concat(F.lit("company:"), F.col("right_row_key").cast("string")).alias("target_node_id"),
        F.lit("MATCHES").alias("edge_type"),
        F.col("final_confidence").alias("weight"),
        F.col("final_decision_source").alias("source"),
        F.col("final_reason").alias("evidence")
    )
)

review_edges = (
    final_df
    .filter(F.col("final_decision") == "REVIEW")
    .select(
        F.concat(F.lit("edge:review:"), F.col("left_row_key").cast("string")).alias("edge_id"),
        F.concat(F.lit("input:"), F.col("left_row_key").cast("string")).alias("source_node_id"),
        F.lit("review:manual_review").alias("target_node_id"),
        F.lit("NEEDS_REVIEW").alias("edge_type"),
        F.col("final_confidence").alias("weight"),
        F.col("final_decision_source").alias("source"),
        F.col("final_reason").alias("evidence")
    )
)

no_match_edges = (
    final_df
    .filter(F.col("final_decision") == "NO_MATCH")
    .select(
        F.concat(F.lit("edge:no_match:"), F.col("left_row_key").cast("string")).alias("edge_id"),
        F.concat(F.lit("input:"), F.col("left_row_key").cast("string")).alias("source_node_id"),
        F.concat(F.lit("company:"), F.col("right_row_key").cast("string")).alias("target_node_id"),
        F.lit("NO_MATCH").alias("edge_type"),
        F.col("final_confidence").alias("weight"),
        F.col("final_decision_source").alias("source"),
        F.col("final_reason").alias("evidence")
    )
)

located_in_edges = (
    final_df
    .filter(F.col("right_country_code").isNotNull())
    .select(
        F.concat(F.lit("edge:located_in:"), F.col("right_row_key").cast("string")).alias("edge_id"),
        F.concat(F.lit("company:"), F.col("right_row_key").cast("string")).alias("source_node_id"),
        F.concat(F.lit("country:"), F.col("right_country_code")).alias("target_node_id"),
        F.lit("LOCATED_IN").alias("edge_type"),
        F.lit(1.0).alias("weight"),
        F.lit("company_country_code").alias("source"),
        F.concat(F.lit("right_country="), F.coalesce(F.col("right_country"), F.lit("")), F.lit("; right_country_code="), F.coalesce(F.col("right_country_code"), F.lit(""))).alias("evidence")
    )
)

top_candidate_edges = (
    final_df
    .select(
        F.concat(F.lit("edge:top_candidate:"), F.col("left_row_key").cast("string")).alias("edge_id"),
        F.concat(F.lit("input:"), F.col("left_row_key").cast("string")).alias("source_node_id"),
        F.concat(F.lit("company:"), F.col("right_row_key").cast("string")).alias("target_node_id"),
        F.lit("HAS_TOP_CANDIDATE").alias("edge_type"),
        F.col("final_confidence").alias("weight"),
        F.col("final_decision_source").alias("source"),
        F.col("final_reason").alias("evidence")
    )
)

edges = (
    match_edges
    .unionByName(review_edges)
    .unionByName(no_match_edges)
    .unionByName(located_in_edges)
    .unionByName(top_candidate_edges)
    .dropDuplicates(["edge_id"])
)

In [0]:
# Save graph nodes and edges as Delta tables.
(
    nodes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nodes_table)
)

(
    edges.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(edges_table)
)

In [0]:
# Review graph node and edge distributions.
display(nodes.groupBy("node_type").count().orderBy(F.desc("count")))
display(edges.groupBy("edge_type").count().orderBy(F.desc("count")))

display(nodes)
display(edges)

In [0]:
# Validate that every edge endpoint has a matching graph node.
edge_node_ids = (
    edges.select(F.col("source_node_id").alias("node_id"))
    .union(edges.select(F.col("target_node_id").alias("node_id")))
    .distinct()
)

missing_edge_nodes = (
    edge_node_ids
    .join(nodes.select("node_id"), on="node_id", how="left_anti")
)

display(missing_edge_nodes)